* Con **PANDAS** es posible leer documentos directos de google drive, carpetas u otros, ademas podemos crear un nuevo dataset a partir de 1 o mas archivos, esto seria un merge de archivos    

en este ejercicio usaremos 2 archivos, haremos todo el proceso:
* carga de datos (pd.read)
* revisar el dataset (head, info y shape)
* sacar estadisticas rapidas (describe).  

paso 2 **(limpieza de datos)**
* manejo de nulos (df.isnull sum() )
* para ver faltantes (df.fillna o dropna)
* duplicados (df.duplicated sum() eliminar con df.drop_duplicates())
* estandarizar mayus ( .str.lower()) o espacios( .str.strip()) o remplazo (.str.replace(antiguo.str.replace(nuevo))
* casteo de datos enteros(.astype(int o str o float))
* casteo seguro (pd.to_numeric) + errors="coerce"
* casteo fecha (pd.to_datetime).  

paso 3 ** cruce de datos (merge / join)**
* identificar la key (columna comun)
* seleccionar el tipo de union (inner(los mas comunes) o left) ejemplo df_final = pd.merge(df1, df2, on='columna_comun', how='inner')   

paso 4 ** transformacion y enriquecimiento**
* columnas calculadas (df['total'] = df['cantidad'] * df['precio'])
* segmentacion (.apply() o pd.cut())
* agrupacion df.groupby('Region')['total'].sum() para obtener los KPIs reales.

paso 5 ** validaciones y consistencia **.
* check de filas compara los numeros de files

paso 6 ** exportacion**
* salida f_final.to_excel('reporte_final.xlsx', index=False) o guardarlo en una base de datos SQL.



**Paso 1: Configuración de los Datasets**.

Imagina que tienes una tabla con las ventas reales y otra con las metas mensuales.

In [2]:
import pandas as pd
import numpy as np

# Dataset 1: Ventas Reales (Con errores de tipo)
df_ventas = pd.DataFrame({
    'Vendedor_ID': ['10', '20', '30', '10', '40'],
    'Venta_Sucia': ['$1.500.000', '$800.000', '$2.100.000', '$900.000', '1.200.000'],
    'Fecha': ['2024-01-01', '2024-01-05', '2024-01-10', '2024-02-01', '2024-01-15']
})

# Dataset 2: Metas (Limpia)
df_metas = pd.DataFrame({
    'Vendedor_ID': [10, 20, 30, 40],
    'Meta_Mensual': [2000000, 1000000, 2500000, 1500000],
    'Nombre': ['Andrés', 'Beatriz', 'Carlos', 'Daniela']
})

**Paso 2: Exploracion y diagnostico**


In [6]:
print("dataset ventas")
df_ventas.head()
df_ventas.info()
df_ventas.describe().round(2)
print("-----------------------------")
print("dataset metas")
df_metas.head()
df_metas.info()
df_metas.describe().round(2)

dataset ventas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Vendedor_ID  5 non-null      object
 1   Venta_Sucia  5 non-null      object
 2   Fecha        5 non-null      object
dtypes: object(3)
memory usage: 252.0+ bytes
-----------------------------
dataset metas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Vendedor_ID   4 non-null      int64 
 1   Meta_Mensual  4 non-null      int64 
 2   Nombre        4 non-null      object
dtypes: int64(2), object(1)
memory usage: 228.0+ bytes


,Vendedor_ID,Meta_Mensual
count,4.00,4.00
mean,25.00,1750000.00
std,12.91,645497.22
min,10.00,1000000.00
25%,17.50,1375000.00
50%,25.00,1750000.00
75%,32.50,2125000.00
max,40.00,2500000.00


**Paso 3: El Proceso de Casteo (Limpieza Crítica)**.

Si intentas hacer el merge ahora, fallará porque Vendedor_ID en una tabla es Texto ('10') y en la otra es Entero (10). Además, no puedes sumar la columna Venta_Sucia porque tiene signos de peso y puntos.

In [7]:
# 1. Limpiamos la columna de dinero y la casteamos a float
df_ventas['Venta_Limpia'] = df_ventas['Venta_Sucia'].str.replace('$', '', regex=False).str.replace('.', '', regex=False)
df_ventas['Venta_Limpia'] = pd.to_numeric(df_ventas['Venta_Limpia'])

# 2. Casteamos la llave de cruce a entero para que coincidan
df_ventas['Vendedor_ID'] = df_ventas['Vendedor_ID'].astype(int)

# 3. Casteamos la fecha
df_ventas['Fecha'] = pd.to_datetime(df_ventas['Fecha'])

**Paso 4: Cruce de Datos (Merge)**   

Ahora que los IDs hablan el mismo "idioma", unimos las tablas.

In [8]:
# Unimos para traer el nombre del vendedor y su meta a la tabla de ventas
df_consolidado = pd.merge(df_ventas, df_metas, on='Vendedor_ID', how='left')

**Paso 5: Creación de Nuevas Columnas (Lógica de Negocio)**

Ahora crearemos columnas que nos den información de valor


In [9]:
# 1. Agrupamos por vendedor para tener el total acumulado
df_resumen = df_consolidado.groupby(['Nombre', 'Meta_Mensual'])['Venta_Limpia'].sum().reset_index()

# 2. Nueva Columna: Diferencia respecto a la meta
df_resumen['Diferencia'] = df_resumen['Venta_Limpia'] - df_resumen['Meta_Mensual']

# 3. Nueva Columna: Porcentaje de cumplimiento
df_resumen['%_Cumplimiento'] = (df_resumen['Venta_Limpia'] / df_resumen['Meta_Mensual'] * 100).round(2)

# 4. Nueva Columna Condicional: Estado (Uso de np.where)
df_resumen['Estado'] = np.where(df_resumen['Venta_Limpia'] >= df_resumen['Meta_Mensual'], 'Meta Cumplida', 'Bajo Meta')

print(df_resumen)

    Nombre  Meta_Mensual  Venta_Limpia  Diferencia  %_Cumplimiento  \
0   Andrés       2000000       2400000      400000           120.0   
1  Beatriz       1000000        800000     -200000            80.0   
2   Carlos       2500000       2100000     -400000            84.0   
3  Daniela       1500000       1200000     -300000            80.0   

          Estado  
0  Meta Cumplida  
1      Bajo Meta  
2      Bajo Meta  
3      Bajo Meta  
